In [ ]:
# Import basic packgaes 
import numpy as np
import pandas as pd
import os
from scipy.signal import argrelextrema

# Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

from scipy.interpolate import interp1d

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [ ]:
directory_name = "D:\\1.5inch Pump Data\\P2_ASP-100,WDP-20\\T02_D2_P2_15_100-20"

In [ ]:
os.chdir(directory_name)

In [ ]:
os.getcwd()

In [ ]:
# Add csv files in the directory to a list
csvs = [x for x in os.listdir(directory_name) if x.endswith('.csv')]

# Add filenames of csv files to a list
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

# Create an empty dictionary
dic_csv = {}

# Load csv files into the dictionary
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [ ]:
# Create a new dictionary for cleaned data
clean_dic_1 = {}

# Load cleaned csv files to the dictionary
for key, df in dic_csv.items():
    clean_dic_1[key] = df[(df['P2-Air Supply FLowrate'] >=50)&(df['P2-Air Supply FLowrate'] <= 110)]

In [ ]:
# Reset index to clean up the dataframes
for key, df in clean_dic_1.items():
    df.reset_index(inplace=True, drop=True)

In [ ]:
# Create a new dictionary for cleaned data
clean_dic = {}

# Load cleaned csv files to the dictionary
for key, df in clean_dic_1.items():
    clean_dic[key] = df[df['P2-Air Supply Pressure'] > 80]

In [ ]:
clean_dic['06_T02_D2_P2_15_100-20_27Mar24']

In [ ]:
# Reset index to clean up the dataframes
for key, df in clean_dic.items():
    df.reset_index(inplace=True, drop=True)

In [ ]:
# Get the number of keys in the dictionary
num_keys = len(clean_dic)

# Specify the starting date
start_date = '2023-03-01 00:00:00.000'

# Generate the list of date elements based on the number of keys
Month_date_list = pd.date_range(start_date, freq='1D', periods=num_keys).strftime('%Y-%m-%d %H:%M:%S.%f').tolist()

In [ ]:
# Create a new column to add the correct date
for (key, df), i in zip(clean_dic.items(),Month_date_list) :    
    df['newtime'] = pd.date_range(i, freq='30ms', periods=len(df))

In [ ]:
# Change newtime to datetime values 
for key, df in clean_dic.items():
    df['newtime'] =pd.to_datetime(df['newtime'])

In [ ]:
# Set newtime as index
for key, df in clean_dic.items():
    df.set_index(['newtime'], inplace=True)

In [ ]:
for key, df in clean_dic.items():
    df['Water_discharge_m3/hr'] = df['P2-Discharge Flowrate']*0.23

In [ ]:
for key, df in clean_dic.items():
    df['Air_Flowrare_m3/min'] = df['P2-Air Supply FLowrate']*0.028

In [ ]:

for key, df in clean_dic.items():
    df['Performance_ratio'] = (df['Water_discharge_m3/hr']/df['Air_Flowrare_m3/min'])*60

In [ ]:
'''''
fig1 = plt.figure(figsize=(40,12))

SMALL_SIZE = 25

for key, df in clean_dic.items():
    plt.plot(df.index, df['Performance_ratio'],color='blue')
    #plt.plot(df.index, df['P2-CPM'],color='magenta')             
    plt.ylim(100,1000)
    #plt.title("P2_IP-100psi_BP-20psi")
    #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.show()
'''''

In [ ]:
P2_Merg_df = pd.concat(clean_dic.values(), ignore_index=True)

In [ ]:
P2_Merg_df.reset_index(inplace=True, drop=True)

In [ ]:
P2_Merg_df.head()

In [ ]:
len(P2_Merg_df)

In [ ]:
time_list = [n for n in np.arange(0,150,(150/len(P2_Merg_df)))]

In [ ]:
len(time_list)

In [ ]:
P2_Merg_df['Diaphragm_life'] = time_list

In [ ]:
plt.figure(figsize=(30,8))
#plt.xticks(np.arange(np.min(P2_Merg_df['Diaphragm_life']),np.max((P2_Merg_df['Diaphragm_life'])),15))
plt.plot(P2_Merg_df['Diaphragm_life'],P2_Merg_df['Performance_ratio'],marker = '.')
plt.xlabel('Hrs')
plt.ylabel('Performance_Ratio')
#plt.ylim(0,5000)
plt.title('Diaphragm lifetime vs Performance_Ratio')
plt.show()

In [ ]:
P2_Merg_df.to_csv(r"D:\\2 inch Pump CSV Files\\T03_20_Perform_ratio_370Hrs.csv")

In [ ]:
# Create a list of csv files of FFT
Perform_ratio_point = []

In [ ]:
# Define the time interval for slicing
start_time = P2_Merg_df.index[0]
end_time = P2_Merg_df.index[-1]
time_interval = 1499400

In [ ]:
current_time = start_time
while current_time <= end_time:
            
    next_time = current_time + time_interval
            
    # Slice the DataFrame based on the specified time range
    sliced_df = P2_Merg_df.iloc[(P2_Merg_df.index >= current_time) & (P2_Merg_df.index <= next_time)]
            
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        # Append the DataFrame to the list
        Perform_ratio_point.append(np.average(sliced_df['Performance_ratio']))                
    else:
        # Handle the case where the DataFrame is empty (e.g., print a message)
        print(f"No data in the time range {current_time} - {next_time}")
    current_time = next_time

In [ ]:
Perf_df = pd.DataFrame()

In [ ]:
Perf_df['Efficiency'] = Perform_ratio_point

In [ ]:
Perf_df.reset_index(inplace=True)

In [ ]:
Perf_df.head()

In [ ]:
len(Perf_df)

In [ ]:
time_list_Df0 = [n for n in np.arange(0,150,(150/len(Perf_df)))]

In [ ]:
len(time_list_Df0)

In [ ]:
Perf_df['Diaphragm_life'] = time_list_Df0

In [ ]:
Max_Eff = argrelextrema(Perf_df['Efficiency'].values, np.greater_equal,order=2)[0]
Perf_df['max_point'] = np.nan
Perf_df.loc[Max_Eff, 'max_point'] = Perf_df.loc[Perf_df.index[Max_Eff], 'Efficiency']

In [ ]:
Perf_df.set_index(['Diaphragm_life'], inplace=True)

In [ ]:
plt.figure(figsize=(25,5))
#plt.xticks(np.arange(np.min(Perf_df.index),np.max((Perf_df.index)),15))
plt.plot(Perf_df.index,Perf_df['Efficiency'],marker = '.')
plt.xlabel('Hrs')
plt.ylabel('Performance_Ratio')
plt.title('Diaphragm lifetime vs Performance_Ratio')
plt.show()

In [ ]:
f = interp1d(x=Perf_df.index,y=Perf_df['Efficiency'],kind=2)
x2 = np.linspace(start=0,stop=131.1,num=10000)
y2 = f(x2)

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(x2, y2, color='b')
plt.plot(Perf_df.index, Perf_df['Efficiency'], ls='', marker='o', color='r')
plt.xlabel('Hrs')
plt.ylabel('Performance_Ratio')
plt.title('Diaphragm lifetime vs Performance_Ratio')
plt.show()

In [ ]:
# set figure size 
plt.figure( figsize=(20,5)) 
#plt.xticks(np.arange(np.min(Perf_df.index),np.max((Perf_df.index)),15))
#plt.ylim(0.2,0.9)
sns.regplot(x=Perf_df.index, y=Perf_df['Efficiency'], lowess=True, line_kws={'color':'red'})
plt.xlabel('Hrs')
plt.ylabel('Performance_Ratio')

In [ ]:
Percentage_list = [600,540,480,420,360,300]

In [ ]:
for i in Percentage_list:
    Perf_df[f"{'T03_Perf_%_cal_'}{i}"] = ((Perf_df['Efficiency']-i)/i)*100

In [ ]:
Perf_df

In [ ]:
Perf_df.drop(columns=['index'], axis=1, inplace=True)

In [ ]:
Perf_df.to_csv(r"D:\\2 inch Pump CSV Files\\T03_P3_20_0C_Perform_rati-P2.csv")

In [ ]:
#Perf_df['cumu_sum_perform'] = Perf_df['Efficiency'].cumsum()

In [ ]:
# set figure size 
#plt.figure(figsize = (30,8)) 
#plt.xticks(np.arange(np.min(Perf_df['Diaphragm_life']),np.max((Perf_df['Diaphragm_life'])),50))
#plt.ylim(0.2,0.9)
#sns.regplot(x=Perf_df['Diaphragm_life'], y=Perf_df['cumu_sum_perform'], lowess=True, line_kws={'color':'red'})

In [ ]:
#Perf_df.to_csv(r"D:\\FFT Analysis results of 2inch pump\\T02_P1_20_10C_1270Hrs_perform_ratio.csv")